# Measurement Technique Merge Candidate Review

This notebook reproduces a cross-ontology merge-candidate review from `measTechOnly_mapped_triples.tsv`.

Outputs:
- `review_candidates`: candidate term pairs with confidence tiers (`exact_norm`, `near_match`)
- optional export to TSV for manual curation


### Import and Display Settings
Imports all required libraries and sets pandas display options so long labels/IRIs are easier to inspect in notebook tables.


In [ ]:
import json
import re
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 160)


### Load Input Data
Define the file paths, load the mapped triples table and the curated best mapping dictionary, and prints basic row/predicate counts as a quick sanity check.


In [ ]:
# Paths
base = Path.cwd()
result_path = base / 'results'
triples_path = result_path / 'measTechOnly_mapped_triples.tsv'
best_map_path = result_path / 'ordered_mapping_best_dict.json'

# If running from project root instead of measTechNet/, uncomment below:
# base = Path.cwd() / 'measTechNet'
# result_path = base / 'results'
# triples_path = result_path / 'measTechOnly_mapped_triples.tsv'
# best_map_path = result_path / 'ordered_mapping_best_dict.json'

df = pd.read_csv(triples_path, sep='\t', index_col=0)
with open(best_map_path, 'r', encoding='utf-8') as f:
    ordered_mapping_best_dict = json.load(f)

print('triples:', len(df))
print('predicate counts:', df['predicate'].value_counts().to_dict())


### Helper Functions
Reusable utilities for:
- label normalization for robust text comparisons
- ontology prefix extraction from IRIs
- sameAs union-find closure so already-linked terms are not re-suggested
- canonical label selection per IRI


In [ ]:
def normalize_label(text: str) -> str:
    text = str(text).lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def ontology_code(iri: str) -> str:
    iri = str(iri)
    m = re.search(r'/obo/([A-Za-z]+)_', iri)
    if m:
        return m.group(1).upper()
    if 'bioassayontology.org/bao' in iri:
        return 'BAO'
    if 'edamontology.org' in iri:
        return 'EDAM'
    if 'ebi.ac.uk/efo' in iri:
        return 'EFO'
    return re.sub(r'^https?://', '', iri).split('/')[0].split('#')[0]

def build_sameas_union_find(dataframe: pd.DataFrame):
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    same = dataframe.loc[dataframe['predicate'].astype(str).str.strip().eq('sameAs'), ['subject_id', 'object_id']]
    for _, row in same.iterrows():
        union(row['subject_id'], row['object_id'])

    return find

def canonical_label_per_id(dataframe: pd.DataFrame):
    labels = defaultdict(list)
    for id_col, label_col in [('subject_id', 'subject'), ('object_id', 'object')]:
        for iri, label in dataframe[[id_col, label_col]].dropna().itertuples(index=False):
            label = str(label).strip()
            if label:
                labels[iri].append(label)
    return {iri: Counter(vals).most_common(1)[0][0] for iri, vals in labels.items()}


### Build Candidate Pairs
Construct candidate merge pairs in two confidence tiers:
- `exact_norm`: identical normalized labels across ontologies
- `near_match`: high-similarity label pairs (for manual review)
It also tags whether each pair is already represented in `ordered_mapping_best_dict`.


In [ ]:
# Build candidate pairs
find = build_sameas_union_find(df)
id_to_label = canonical_label_per_id(df)

records = []
for iri, label in id_to_label.items():
    nlabel = normalize_label(label)
    tokens = nlabel.split()
    records.append((iri, label, nlabel, tokens, ontology_code(iri)))

# Tier 1: exact normalized label matches across ontologies
by_norm = defaultdict(list)
for rec in records:
    if rec[2]:
        by_norm[rec[2]].append(rec)

generic_only = {
    'assay', 'method', 'methods', 'technique', 'techniques',
    'test', 'analysis', 'measurement', 'measurements', 'procedure', 'process'
}

exact_rows = []
for nlabel, items in by_norm.items():
    if len(items) < 2:
        continue
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            iri1, label1, _, tokens1, onto1 = items[i]
            iri2, label2, _, tokens2, onto2 = items[j]

            if onto1 == onto2:
                continue
            if find(iri1) == find(iri2):
                continue
            if len(set(tokens1)) == 1 and list(set(tokens1))[0] in generic_only:
                continue

            # Light down-weight for single-token labels
            score = 0.70 if len(tokens1) <= 1 else 1.00

            exact_rows.append({
                'confidence_tier': 'exact_norm',
                'confidence_score': round(score, 3),
                'normalized_label': nlabel,
                'label_1': label1,
                'id_1': iri1,
                'ontology_1': onto1,
                'label_2': label2,
                'id_2': iri2,
                'ontology_2': onto2,
                'in_ordered_mapping_best_dict': bool(ordered_mapping_best_dict.get(iri1) == iri2 or ordered_mapping_best_dict.get(iri2) == iri1)
            })

# Tier 2: high-similarity near matches (for manual review)
by_first_token = defaultdict(list)
for rec in records:
    if len(rec[3]) >= 2:
        by_first_token[rec[3][0]].append(rec)

near_rows = []
for first_token, items in by_first_token.items():
    # safety bound to avoid O(n^2) blowups on very large buckets
    if len(items) > 250:
        continue

    for i in range(len(items)):
        iri1, label1, nlabel1, tok1, onto1 = items[i]
        if len(tok1) < 2:
            continue
        for j in range(i + 1, len(items)):
            iri2, label2, nlabel2, tok2, onto2 = items[j]

            if onto1 == onto2:
                continue
            if find(iri1) == find(iri2):
                continue
            if nlabel1 == nlabel2:
                continue
            if abs(len(nlabel1) - len(nlabel2)) > 8:
                continue

            sim = SequenceMatcher(None, nlabel1, nlabel2).ratio()
            if sim < 0.93:
                continue

            # Require moderate token overlap
            s1, s2 = set(tok1), set(tok2)
            jaccard = len(s1 & s2) / len(s1 | s2) if (s1 | s2) else 0.0
            if jaccard < 0.50:
                continue

            near_rows.append({
                'confidence_tier': 'near_match',
                'confidence_score': round(sim, 3),
                'normalized_label': f'{nlabel1} || {nlabel2}',
                'label_1': label1,
                'id_1': iri1,
                'ontology_1': onto1,
                'label_2': label2,
                'id_2': iri2,
                'ontology_2': onto2,
                'in_ordered_mapping_best_dict': bool(ordered_mapping_best_dict.get(iri1) == iri2 or ordered_mapping_best_dict.get(iri2) == iri1)
            })


### Build `review_candidates` Output
Output the results-- Combine and deduplicate candidate rows, apply measurement/methodology-focused keyword filters, sort candidates, and report tier counts.


In [ ]:
# Combine and deduplicate by unordered pair + tier
all_rows = exact_rows + near_rows
seen = set()
deduped = []
for row in sorted(all_rows, key=lambda r: (-r['confidence_score'], r['confidence_tier'], r['ontology_1'], r['ontology_2'])):
    pair_key = tuple(sorted((row['id_1'], row['id_2']))) + (row['confidence_tier'],)
    if pair_key in seen:
        continue
    seen.add(pair_key)
    deduped.append(row)

review_candidates = pd.DataFrame(deduped)

# Domain-focused filter to prioritize measurement/methodology terms
keep_keywords = (
    'sequenc', 'microscop', 'spectroscop', 'chromatograph', 'electrophores',
    'assay', 'immuno', 'imaging', 'tomograph', 'centrifug', 'elisa',
    'cytometr', 'proteom', 'metabolom', 'transcriptom'
)
exclude_keywords = (
    'surgery', 'transplant', 'amputation', 'oophorectomy', 'appendectomy',
    'contraception', 'treatment', 'autopsy'
)

if not review_candidates.empty:
    norm_text = review_candidates['normalized_label'].str.lower()
    keep_mask = norm_text.str.contains('|'.join(keep_keywords), regex=True)
    exclude_mask = norm_text.str.contains('|'.join(exclude_keywords), regex=True)
    review_candidates = review_candidates.loc[keep_mask & ~exclude_mask].copy()

review_candidates = review_candidates.sort_values(
    by=['confidence_tier', 'confidence_score', 'normalized_label'],
    ascending=[True, False, True]
).reset_index(drop=True)

print('review_candidates:', len(review_candidates))
print(review_candidates['confidence_tier'].value_counts().to_dict())
review_candidates.head(50)


### Focus on New Candidates
Identify `new_candidates` by excluding pairs already present in `ordered_mapping_best_dict`, so review effort is focused on unmapped merges.


In [ ]:
# Optional: keep only candidates not already represented in ordered_mapping_best_dict
new_candidates = review_candidates.loc[~review_candidates['in_ordered_mapping_best_dict']].copy()
print('new_candidates:', len(new_candidates))
new_candidates.head(50)


### Export results
Export tab-separated outputs for manual curation:
- `review_candidates.tsv`
- `review_candidates_new_only.tsv`


In [ ]:
# Optional export for manual curation
out_all = result_path / 'review_candidates.tsv'
out_new = result_path / 'review_candidates_new_only.tsv'
review_candidates.to_csv(out_all, sep='\t', index=False)
new_candidates.to_csv(out_new, sep='\t', index=False)
print('wrote:', out_all)
print('wrote:', out_new)
